## Silver to Gold

In [9]:
# FUNCTIONS & PACKAGES
import json
from datetime import datetime
import pytz
import re

CURRENT_YEAR = 2026

# Importing raw files from silver
def import_silver(file_name):
    with open(f"../data/silver/{file_name}", "r") as f:
        raw_data = json.load(f)
    return raw_data

# Exporting clean files to gold
def export_gold(file_name, clean_data, indent_num):
    with open(f"../data/gold/{file_name}", "w") as f:
        json.dump(clean_data, f, indent=indent_num, ensure_ascii=False)
    print(f"Merge complete. Output saved to ../data/gold/{file_name}")

# Set the type of each variable
def cast_record(record, type_map):
    for field, field_type in type_map.items():
        value = record.get(field)

        if value in (None, ""):
            record[field] = None
        else:
            try:
                record[field] = field_type(value)
            except (ValueError, TypeError):
                record[field] = None

    return record

In [2]:
# CLEAN MEMBER DATA


# Load raw JSON member data
raw_members = import_silver("members_119.json")

clean_members = []

# Loop through pages
for page in raw_members.get("pages", []):
    for member in page.get("members", []):
        
        bioguide_id = member.get("bioguideId")
        party = member.get("partyName")[0]
        state_name = member.get("state")
        district = member.get("district")

        # Get full name (first middle last)
        full_name = member.get("name")        
        last, first = full_name.split(",", 1)
        first = first.strip()
        last = last.strip()
        name = f"{first} {last}"

        # Determine terms spent in Congress
        terms = member.get("terms", {}).get("item", [])
        total_years = 0
        for term in terms:
            start_year = term.get("startYear")
            end_year = term.get("endYear")
            if start_year:
                if end_year:
                    total_years += int(end_year) - int(start_year)
                else:
                    total_years += CURRENT_YEAR - int(start_year)
        years_in_congress = total_years

        # Extract chamber
        chamber = None
        if terms:
            latest_term = terms[-1]
            chamber_raw = latest_term.get("chamber")
            if chamber_raw == "House of Representatives":
                chamber = "H"
            elif chamber_raw == "Senate":
                chamber = "S"
            else:
                chamber = chamber_raw

        # Account for at-large districts
        if chamber == "H":
            if district in (None, "", "At-Large"):
                district = 0

        clean_member = {
            "member_id": bioguide_id,
            "full_name": full_name,
            "party": party,
            "state_name": state_name,
            "district": district,
            "chamber": chamber,
            "years_in_congress": years_in_congress
        }

        clean_members.append(clean_member)

# Load raw JSON member bios data
raw_bio_data = import_silver("member_bios_119.json")

clean_bios = []

for member_bio in raw_bio_data:
    member_id = member_bio.get("id")
    first_name = member_bio.get("unaccentedGivenName")
    last_name = member_bio.get("unaccentedFamilyName")
    birth_year = member_bio.get("birthYear")

    # Calculate age safely
    if birth_year:
        age = CURRENT_YEAR - int(birth_year)
    else:
        age = None

    clean_member_bio = {
        "member_id": member_id,
        "first_name": first_name,
        "last_name": last_name,
        "age": age
    }

    clean_bios.append(clean_member_bio)


# Convert bios list into dictionary for fast lookup
bios_lookup = {bio["member_id"]: bio for bio in clean_bios}

merged_members = []

for member in clean_members:
    member_id = member["member_id"]

    # Get matching bio (if exists)
    bio = bios_lookup.get(member_id, {})

    merged_member = {
        "member_id": member_id,
        "full_name": member.get("full_name"),
        "first_name": bio.get("first_name"),
        "last_name": bio.get("last_name"),
        "party": member.get("party"),
        "chamber": member.get("chamber"),
        "state_name": member.get("state_name"),
        "district": member.get("district"),
        "years_in_congress": member.get("years_in_congress"),
        "age": bio.get("age")
    }

    merged_members.append(merged_member)

# Make sure data type is correct
TYPE_MAP_MEMBERS = {
    "member_id": str,
    "full_name": str,
    "first_name": str,
    "last_name": str,
    "party": str,
    "state_name": str,
    "chamber": str,
    "district": int,
    "years_in_congress": int,
    "age": int,
}

merged_members = [cast_record(r, TYPE_MAP_MEMBERS) for r in merged_members]

# Check data:
print("Total records:", len(merged_members))
print("Unique IDs:", len(set(r["member_id"] for r in merged_members)))


# Write merged gold JSON
export_gold("members_119.json", merged_members, 4)


Total records: 538
Unique IDs: 538
Merge complete. Output saved to ../data/gold/members_119.json


In [ ]:
# CLEAN BILL DATA

data = import_silver("bills_119.json")
data_policy_area = import_silver("bills_policy_area_119.json")

cleaned_bills = []

# Lookup policy area
policy_lookup = {}

for page in data_policy_area.get("pages", []):
    for bill in page.get("bills", []):
        congress = bill.get("congress")
        bill_type = bill.get("type")
        number = bill.get("number")
        policy_area = bill.get("policyArea", {}).get("name")

        bill_id = f"{bill_type}{number}_{congress}"
        policy_lookup[bill_id] = policy_area

# Flatten pages -> bills
for page in data.get("pages", []):
    for bill in page.get("bills", []):

        congress = bill.get("congress")
        chamber_raw = bill.get("originChamber")[0]
        bill_type = bill.get("type")
        number = bill.get("number")
        title = bill.get("title")

        policy_area = policy_lookup.get(bill_id)
        
        # Construct bill_id
        bill_id = f"{bill_type}{number}_{congress}"

        cleaned_bills.append({
            "bill_id": bill_id,
            "bill_type": bill_type,
            "bill_number": number,
            "congress": congress,
            "chamber": chamber,
            "title": title,
            "policy_area": policy_area
        })

# Make sure data type is correct
TYPE_MAP_BILLS = {
    "bill_id": str,
    "bill_type": str,
    "bill_number": int,
    "congress": int,
    "chamber": str,
    "title": str,
    "policy_area": str,
}

cleaned_bills = [cast_record(r, TYPE_MAP_BILLS) for r in cleaned_bills]

# Check data:
print("Total records:", len(cleaned_bills))
print("Unique IDs:", len(set(r["bill_id"] for r in cleaned_bills)))

# Save cleaned data
export_gold("bills_119.json", cleaned_bills, 4)




Total records: 13250
Unique IDs: 13250
Merge complete. Output saved to ../data/gold/bills_119.json


In [ ]:
# CLEAN AMENDMENT DATA

data = import_silver("amendments_119.json")

cleaned_amendments = []

for page in data.get("pages", []):
    for amendment in page.get("amendments", []):

        congress = amendment.get("congress")
        amendment_type = amendment.get("type")
        number = amendment.get("number")

        # Build amendment_id
        amendment_id = f"{amendment_type}{number}_{congress}"

        # Determine chamber
        if amendment_type == "SAMDT":
            chamber = "S"
        elif amendment_type == "HAMDT":
            chamber = "H"
        else:
            chamber = None

        # Prefer description, fallback to purpose
        purpose = amendment.get("description") or amendment.get("purpose")

        cleaned_amendments.append({
            "amendment_id": amendment_id,
            "amendment_type": amendment_type,
            "congress" : congress,
            "chamber" : chamber,
            "purpose": purpose
        })

# Make sure data type is correct
TYPE_MAP_AMEDMENTS = {
    "amendment_id": str,
    "amendment_type": str,
    "congress": int,
    "chamber": str,
    "purpose": str,
}

cleaned_amendments = [cast_record(r, TYPE_MAP_AMEDMENTS) for r in cleaned_amendments]

# Check data:
print("Total records:", len(cleaned_amendments))
print("Unique IDs:", len(set(r["amendment_id"] for r in cleaned_amendments)))

export_gold("amendments_119.json", cleaned_amendments, 2)



Total records: 4455
Unique IDs: 4455
Merge complete. Output saved to ../data/gold/amendments_119.json


In [17]:
# CLEAN VOTE DATA

# ADDING 'question' FIELD TO 'house_rollcall_119.json'
import json
def adding_question_field(house_rollcall_data):
    party_totals_files = [
        "../data/silver/house_vote_party_totals_119_1.json",
        "../data/silver/house_vote_party_totals_119_2.json"
    ]

    rollcall_file = house_rollcall_data

    # 1) Build identifier -> voteQuestion mapping
    identifier_to_question = {}

    for fpath in party_totals_files:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        for page in data.get("pages", []):
            vote = page.get("houseRollCallVote")
            if (
                vote
                and vote.get("identifier")
                and vote.get("voteQuestion")
            ):
                identifier_to_question[vote["identifier"]] = vote["voteQuestion"]

    # 2) Load rollcall data
    with open(rollcall_file, "r", encoding="utf-8") as f:
        rollcall_data = json.load(f)

    updates = 0
    skipped_existing = 0

    # 3) Add 'voteQuestion' only if missing or empty
    for page in rollcall_data.get("pages", []):
        for vote in page.get("houseRollCallVotes", []):
            ident = vote.get("identifier")

            if not ident:
                continue

            # Skip if voteQuestion already exists and is non-empty
            existing_value = vote.get("voteQuestion")
            if existing_value and str(existing_value).strip():
                skipped_existing += 1
                continue

            if ident in identifier_to_question:
                vote["voteQuestion"] = identifier_to_question[ident]
                updates += 1

    # 4) Save only if changes were made
    if updates > 0:
        with open(rollcall_file, "w", encoding="utf-8") as f:
            json.dump(rollcall_data, f, indent=2)

adding_question_field("../data/silver/house_rollcall_119.json")

def clean_house_votes(input_path):
    data = import_silver(input_path)

    cleaned_votes = []
    seen_ids = set()

    for page in data.get("pages", []):
        for vote in page.get("houseRollCallVotes", []):

            congress = vote.get("congress")
            session = vote.get("sessionNumber")
            roll_number = vote.get("rollCallNumber")
            leg_type = vote.get("legislationType")
            leg_num = vote.get("legislationNumber")
            question = vote.get("voteQuestion")

            vote_id = f"roll_H{str(roll_number).zfill(5)}_{congress}_{session}"

            bill_id = f"{leg_type}{leg_num}_{congress}"

            # EXCLUDING AMENDMENTS (fix later if you want amendments)
            if leg_type == None:
                continue

            if vote_id in seen_ids:
                continue
            seen_ids.add(vote_id)


            cleaned_votes.append({
                "vote_id": vote_id,
                "bill_id": bill_id,
                "question": question,
                "chamber": "H",
                "congress": congress,
                "session_num": session,
                "vote_date": vote.get("startDate"),
                "result": vote.get("result")
            })

    # Make sure data type is correct
    TYPE_MAP_VOTES = {
        "vote_id": str,
        "bill_id": str,
        "question": str,
        "chamber": str,
        "congress": int,
        "session_num": int,
        "vote_date": str,
        "result": str
    }

    cleaned_votes = [cast_record(r, TYPE_MAP_VOTES) for r in cleaned_votes]

    # Check data:    
    print("House votes")
    print("Total records:", len(cleaned_votes))
    print("Unique IDs:", len(set(r["vote_id"] for r in cleaned_votes)))

    return cleaned_votes

def clean_senate_votes(input_path):
    data = import_silver(input_path)

    cleaned_votes = []

    vote_summary = list(data.values())[0]["vote_summary"]

    congress = int(vote_summary.get("congress"))
    session = int(vote_summary.get("session"))

    for vote in vote_summary.get("votes", {}).get("vote", []):

        vote_number = vote.get("vote_number")
        vote_id = f"roll_S{vote_number}_{congress}_{session}"

        leg_type, leg_num = normalize_senate_issue(vote.get("issue"))

        if leg_type == None:
                continue
        
        # Extract question
        q = vote.get("question")
        if isinstance(q, dict):
            question = q.get("#text", "")
        elif isinstance(q, str):
            question = q
        else:
            question = ""

        # Construct bill_id
        bill_id = f"{leg_type}{leg_num}_{congress}"

        vote_date = normalize_senate_date(
            vote.get("vote_date"),
            congress,
            session
        )


        cleaned_votes.append({
            "vote_id": vote_id,
            "bill_id": bill_id,
            "question": question,
            "chamber": "S",
            "congress": congress,
            "session_num": session,
            "vote_date": vote_date,
            "result": vote.get("result")
        })
        
    # Make sure data type is correct
    TYPE_MAP_VOTES = {
        "vote_id": str,
        "bill_id": str,
        "question": str,
        "chamber": str,
        "congress": int,
        "session_num": int,
        "vote_date": str,
        "result": str
    }

    cleaned_votes = [cast_record(r, TYPE_MAP_VOTES) for r in cleaned_votes]

    # Check data:    
    print("Senate votes")
    print("Total records:", len(cleaned_votes))
    print("Unique IDs:", len(set(r["vote_id"] for r in cleaned_votes)))

    return cleaned_votes

# Dealing with nomination vote data
def normalize_senate_issue(issue):
    if not issue:
        return None, None

    # Remove periods and extra whitespace
    cleaned = issue.replace(".", "").strip()

    # PN case (nominations)
    if cleaned.startswith("PN"):
        number = cleaned[2:].strip()
        # return "PN", number

        # REMOVING NOMINATIONS (fix later if wanting nominations)
        return None, None

    # Handle SRes / SJRes + number, remove space
    match_special = re.match(r"(S(?:J)?RES)\s*(\d+[-\d]*)", cleaned, re.IGNORECASE)
    if match_special:
        leg_type = match_special.group(1).upper()  # Ensure uppercase
        leg_number = match_special.group(2)
        return leg_type, leg_number

    # General legislation pattern (letters + number)
    match = re.match(r"([A-Z]+)\s*(\d+[-\d]*)", cleaned, re.IGNORECASE)
    if match:
        leg_type = match.group(1).upper()
        leg_number = match.group(2)
        return leg_type, leg_number

    # Fallback (procedural / unknown)
    return cleaned, None

# Converting senate date to timestamp format
def normalize_senate_date(raw_date, congress, session):
    if not raw_date:
        return None

    # Map congress/session to calendar year
    base_year = 1789 + (congress - 1) * 2
    year = base_year if session == 1 else base_year + 1

    dt = datetime.strptime(f"{raw_date}-{year}", "%d-%b-%Y")

    eastern = pytz.timezone("US/Eastern")
    dt = eastern.localize(dt.replace(hour=12))

    return dt.isoformat()


def build_gold_votes(house_path, senate_path):

    house_votes = clean_house_votes(house_path)
    senate_votes = clean_senate_votes(senate_path)

    all_votes = house_votes + senate_votes

    export_gold("votes_119.json", all_votes, 2)


build_gold_votes("house_rollcall_119.json", "senate_rollcall_119.json")


House votes
Total records: 357
Unique IDs: 357
Senate votes
Total records: 303
Unique IDs: 303
Merge complete. Output saved to ../data/gold/votes_119.json


In [ ]:
# CLEAN VOTE RECORD DATA
import json
from pathlib import Path

def clean_vote_records(file_paths):
    cleaned_records = []

    for file_path in file_paths:
        data = import_silver(file_path)

        congress = 119
            
        # Determine chamber & session
        if file_path.startswith("senate"):
            chamber = "S"  # Senate
        else:
            chamber = "H"  # House
        session = int(file_path.split("_")[-1].replace(".json", ""))

        for vote in data.get("votes", []):
            roll_number = str(vote.get("roll_number"))
            vote_id = f"roll_{chamber}{roll_number.zfill(5)}_{congress}_{session}"

            cleaned_records.append({
                "vote_id": vote_id,
                "member_id": vote.get("member_id"),
                "position": vote.get("vote")
            })

    # Deduplication
    unique_records = { (r["vote_id"], r["member_id"]) : r for r in cleaned_records }
    cleaned_records = list(unique_records.values())

    # Make sure data type is correct
    TYPE_MAP_VOTE_RECORDS = {
        "vote_id": str,
        "member_id": str,
        "position": str
    }

    cleaned_records = [cast_record(r, TYPE_MAP_VOTE_RECORDS) for r in cleaned_records]

    # Check data:    
    print("Total records:", len(cleaned_records))
    unique_combinations = set((r["vote_id"], r["member_id"]) for r in cleaned_records)
    print("Unique IDs:", len(unique_combinations))

    return cleaned_records


files = [
    "house_votes_119_1.json",
    "house_votes_119_2.json",
    "senate_votes_119_1.json",
    "senate_votes_119_2.json"
]

vote_records = clean_vote_records(files)

# Save to gold JSON
export_gold("vote_records_119.json", vote_records, 2)


Total records: 260568
Unique IDs: 260568
Merge complete. Output saved to ../data/gold/vote_records_119.json


In [ ]:
# Clean vote total data
import json

def clean_vote_party_totals(file_paths):
    cleaned_records = []

    for file_path in file_paths:
        data = import_silver(file_path)
        if file_path.startswith("house"):
            # Iterate through each page
            for page in data.get("pages", []):
                vote_data = page.get("houseRollCallVote", {})

                congress = str(vote_data.get("congress"))
                session = int(file_path.split("_")[-1].replace(".json", ""))
                roll_number = str(vote_data.get("rollCallNumber")).zfill(5)
                chamber = "H"                
                vote_id = f"roll_{chamber}{roll_number.zfill(5)}_{congress}_{session}"

                for party_total in vote_data.get("votePartyTotal", []):
                    party = party_total.get("voteParty")
                    # Skip records where party is null
                    if party is None:
                        continue

                    record = {
                        "vote_id": vote_id,
                        "party": party,
                        "yes_count": party_total.get("yeaTotal", 0),
                        "no_count": party_total.get("nayTotal", 0),
                        "present_count": party_total.get("presentTotal", 0),
                        "not_voting_count": party_total.get("notVotingTotal", 0)
                    }
                    cleaned_records.append(record)
        if file_path.startswith("senate"):
            for record in data:
                cleaned_records.append({
                    "vote_id": record.get("vote_id"),
                    "party": record.get("party"),
                    "yes_count": record.get("yes_count", 0),
                    "no_count": record.get("no_count", 0),
                    "present_count": record.get("present_count", 0),
                    "not_voting_count": record.get("not_voting_count", 0)
                })
    # Check data:    
    print("Total records:", len(cleaned_records))
    unique_combinations = set((r["vote_id"], r["party"]) for r in cleaned_records)
    print("Unique IDs:", len(unique_combinations))

    return cleaned_records


files = [
    "house_vote_party_totals_119_1.json",
    "house_vote_party_totals_119_2.json",
    "senate_vote_party_totals_119_1.json",
    "senate_vote_party_totals_119_2.json"
]

vote_records = clean_vote_party_totals(files)

# Save to gold JSON
export_gold("vote_party_totals_119.json", vote_records, 4)

Total records: 3420
Unique IDs: 3420
Merge complete. Output saved to ../data/gold/vote_party_totals_119.json


In [ ]:
# ADDING 'question' FIELD TO 'house_rollcall_119.json'
import json

party_totals_files = [
    "../data/silver/house_vote_party_totals_119_1.json",
    "../data/silver/house_vote_party_totals_119_2.json"
]

rollcall_file = "../data/silver/house_rollcall_119.json"

# 1) Build identifier -> voteQuestion mapping
identifier_to_question = {}

for fpath in party_totals_files:
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for page in data.get("pages", []):
        vote = page.get("houseRollCallVote")
        if vote and "identifier" in vote and "voteQuestion" in vote:
            identifier_to_question[vote["identifier"]] = vote["voteQuestion"]

# 2) Load rollcall data
with open(rollcall_file, "r", encoding="utf-8") as f:
    rollcall_data = json.load(f)

# 3) Add 'voteQuestion' to matching entries
for page in rollcall_data.get("pages", []):
    for vote in page.get("houseRollCallVotes", []):
        ident = vote.get("identifier")
        if ident in identifier_to_question:
            vote["voteQuestion"] = identifier_to_question[ident]

# 4) Save updated rollcall file
with open(rollcall_file, "w", encoding="utf-8") as f:
    json.dump(rollcall_data, f, indent=2)